In [15]:
import os
import ee
import geemap
from dotenv import load_dotenv

load_dotenv()

ee_project = os.getenv('EE_PROJECT_ID')
if not ee_project:
    raise ValueError("EE_PROJECT_ID not set in .env file")

ee.Initialize(project=ee_project)


In [16]:
campus_geojson = {
    "type": "Polygon",
    "coordinates": [
        [
            [80.01710357666015, 23.173962177472703],
            [80.03259601593017, 23.165361215115187],
            [80.03654422760009, 23.172502420044232],
            [80.026802444458, 23.181694681000845],
            [80.01542987823485, 23.176960548201308],
        ]
    ]
}
campus = ee.Geometry(campus_geojson)

In [17]:
area = campus.area()
print("Campus Area (m²):", area.getInfo())
print("Campus Area (km²):", area.divide(1e6).getInfo())

Campus Area (m²): 1995406.3843647253
Campus Area (km²): 1.9954063843645904


In [18]:
import math

targeted_area_sqm = float(area.getInfo())

point = {
    "type": "Point",
    "coordinates": [79.88361463362526,23.142324404777362]
}


centre = ee.Geometry(point)


circle_radius = math.sqrt(targeted_area_sqm / math.pi)

#region of interest
roi = centre.buffer(circle_radius)

In [19]:
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud = 1 << 10
    cirrus = 1 << 11
    mask = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    return image.updateMask(mask).divide(10000)

dataset = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(roi)
           .filterDate('2025-11-15', '2026-01-15')
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
           .map(mask_s2_clouds))

new_image = dataset.median().clip(roi)

In [20]:
Map = geemap.Map()

Map.centerObject(roi, 16)

Map.addLayer(new_image, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3,
}, 'Test Area')

Map

Map(center=[23.142325222371163, 79.88361465828382], controls=(WidgetControl(options=['position', 'transparent_…

In [21]:
ndvi = new_image.normalizedDifference(['B8', 'B4']).rename('NDVI')
new_image = new_image.addBands(ndvi)
bands = ['B4', 'B8', 'NDVI']

In [22]:
%run ../svm/forest_classification.ipynb

[[27, 1], [0, 22]]
Accuracy: 0.98
Kappa: 0.9596122778675282


In [23]:
new_classified = new_image.select(bands).classify(classifier)
new_smooth_classification = new_classified.focalMode(1)

In [24]:
Map = geemap.Map()
Map.centerObject(roi, 16)

Map.addLayer(new_image, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3
}, 'Test Area RGB')

Map.addLayer(new_smooth_classification, {
    'min': 0, 
    'max': 1, 
    'palette': ['lightgray', 'darkgreen']
}, 'Test Area Forest Map')

Map

Map(center=[23.142325222371163, 79.88361465828382], controls=(WidgetControl(options=['position', 'transparent_…

In [25]:
test_forest_points = ee.FeatureCollection('users/cosypix/test_forest_points')
test_non_forest_points = ee.FeatureCollection('users/cosypix/test_non_forest_points')

test_points = test_forest_points.merge(test_non_forest_points)

In [26]:
# Count number of features in the forest and non-forest collections
def _count(fc):
    try:
        return int(fc.size().getInfo())
    except Exception as e:
        print("Error counting collection:", e)
        return None

test_forest_count = _count(test_forest_points) if 'forest_points' in locals() else None
test_non_forest_count = _count(test_non_forest_points) if 'non_forest_points' in locals() else None

print("Forest points:", test_forest_count)
print("Non-forest points:", test_non_forest_count)

Forest points: 62
Non-forest points: 62


In [27]:
validation_data = new_image.select(bands).sampleRegions(
    collection=test_points,
    properties=['label'],
    scale=10
)

validation_data = validation_data.filter(ee.Filter.notNull(bands + ['label']))

validated = validation_data.classify(classifier)

In [28]:

confusion_matrix = validated.errorMatrix(
    actual='label',
    predicted='classification'
)

print("Confusion Matrix: ", confusion_matrix.getInfo())
print("Test Area Accuracy: ", confusion_matrix.accuracy().getInfo())
print("Test Area Kappa: ", confusion_matrix.kappa().getInfo())

Confusion Matrix:  [[45, 17], [2, 60]]
Test Area Accuracy:  0.8467741935483871
Test Area Kappa:  0.6935483870967742
